# LangChain + CoordiNode: Graph Chain

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/structured-world/coordinode-python/blob/main/demo/notebooks/02_langchain_graph_chain.ipynb)

Demonstrates `CoordinodeGraph` as a Knowledge Graph backend for LangChain.

**What works right now:**
- `graph.query()` — arbitrary Cypher pass-through
- `graph.schema` / `refresh_schema()` — live graph schema
- `add_graph_documents()` — add Nodes + Relationships from a LangChain `GraphDocument`
- `GraphCypherQAChain` — LLM generates Cypher from a natural-language question *(requires `OPENAI_API_KEY`)*

**Environments:**
- **Google Colab** — uses `coordinode-embedded` (in-process Rust engine, no server needed). Installs pre-built wheel from PyPI (~30 sec).
- **Local / Docker Compose** — connects to a running CoordiNode server via gRPC.

## Install dependencies

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

pkgs = [
    "coordinode",
    "langchain-coordinode",
    "langchain",
    "langchain-community",
    "langchain-openai",
    "nest_asyncio",
]
if IN_COLAB and not os.environ.get("COORDINODE_ADDR"):
    pkgs = ["coordinode-embedded"] + pkgs

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + pkgs,
    check=True,
    timeout=120,
)

import nest_asyncio

nest_asyncio.apply()

print("SDK installed")


## Adapter for embedded mode

`LocalClient` (embedded engine) has the same `.cypher()` API as `CoordinodeClient`.
The `_EmbeddedAdapter` below adds the extra methods that `CoordinodeGraph` expects
when it receives a pre-built `client=` object.

In [ ]:
class _EmbeddedAdapter:
    """Thin wrapper around LocalClient that adds CoordinodeClient-compatible methods."""

    def __init__(self, local_client):
        self._lc = local_client

    def cypher(self, query, params=None):
        return self._lc.cypher(query, params or {})

    def get_schema_text(self):
        lbls = self._lc.cypher("MATCH (n) UNWIND labels(n) AS lbl RETURN DISTINCT lbl ORDER BY lbl")
        rels = self._lc.cypher("MATCH ()-[r]->() RETURN DISTINCT type(r) AS t ORDER BY t")
        lines = ["Node labels:"]
        for r in lbls:
            lines.append(f"  - {r['lbl']}")
        lines.append("\nEdge types:")
        for r in rels:
            lines.append(f"  - {r['t']}")
        return "\n".join(lines)

    # Vector search not available in embedded mode — requires running CoordiNode server.

    def close(self):
        self._lc.close()


## Connect to CoordiNode

In [ ]:
import os, tempfile

# Persistent embedded DB path. Colab has /content which persists across cell
# reruns within a runtime session; locally fall back to the OS temp dir
# (portable across Linux/macOS/Windows). Override via COORDINODE_EMBEDDED_PATH.
COORDINODE_EMBEDDED_PATH = os.environ.get(
    "COORDINODE_EMBEDDED_PATH",
    "/content/coordinode-demo.db"
    if os.path.isdir("/content")
    else os.path.join(tempfile.gettempdir(), "coordinode-demo.db"),
)

if os.environ.get("COORDINODE_ADDR"):
    COORDINODE_ADDR = os.environ["COORDINODE_ADDR"]
    from coordinode import CoordinodeClient

    _cc = CoordinodeClient(COORDINODE_ADDR)
    if not _cc.health():
        _cc.close()
        raise RuntimeError(f"Health check failed for {COORDINODE_ADDR}")
    print(f"Connected to {COORDINODE_ADDR}")
    client = _cc
else:
    # No explicit server — use the embedded in-process engine backed by a file
    # so the graph persists across cell reruns and between sibling demo
    # notebooks within the same runtime.
    try:
        from coordinode_embedded import LocalClient
    except ImportError as exc:
        raise RuntimeError(
            "coordinode-embedded is not installed. "
            "In Colab, rerun the install cell above. "
            "Locally, install from source: "
            "pip install 'git+https://github.com/structured-world/coordinode-python.git#subdirectory=coordinode-embedded' "
            "(requires Rust toolchain, ~5 min build). "
            "Alternatively start a CoordiNode server and set COORDINODE_ADDR."
        ) from exc

    _lc = LocalClient(COORDINODE_EMBEDDED_PATH)
    # Wrap in _EmbeddedAdapter so CoordinodeGraph/PropertyGraphStore can call
    # get_schema_text() — LocalClient has .cypher() only.
    client = _EmbeddedAdapter(_lc)
    print(f"Using embedded LocalClient at {COORDINODE_EMBEDDED_PATH}")


## Create the graph store

Pass the pre-built `client=` object so the store doesn't open a second connection.

In [ ]:
import os, uuid
from langchain_coordinode import CoordinodeGraph
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship
from langchain_core.documents import Document

graph = CoordinodeGraph(client=client)
print("Connected. Schema preview:")
print(graph.schema[:300])

## 1. add_graph_documents

LangChain `GraphDocument` wraps nodes and relationships from an LLM extraction pipeline.
Here we build one manually to show the insert path.

In [ ]:
tag = uuid.uuid4().hex

nodes = [
    Node(id=f"Turing-{tag}", type="Scientist", properties={"born": 1912, "field": "computer science"}),
    Node(id=f"Shannon-{tag}", type="Scientist", properties={"born": 1916, "field": "information theory"}),
    Node(id=f"Cryptography-{tag}", type="Field", properties={"era": "modern"}),
]
rels = [
    Relationship(source=nodes[0], target=nodes[2], type="FOUNDED", properties={"year": 1936}),
    Relationship(source=nodes[1], target=nodes[2], type="CONTRIBUTED_TO"),
    Relationship(source=nodes[0], target=nodes[1], type="INFLUENCED"),
]
doc = GraphDocument(nodes=nodes, relationships=rels, source=Document(page_content="Turing and Shannon"))
graph.add_graph_documents([doc])
print("Documents added")

## 2. query — direct Cypher

In [ ]:
rows = graph.query(
    "MATCH (s:Scientist)-[r]->(f:Field)"
    " WHERE s.name STARTS WITH $prefix"
    " RETURN s.name AS scientist, type(r) AS relation, f.name AS field",
    params={"prefix": f"Turing-{tag}"},
)
print("Scientists → Fields:")
for r in rows:
    print(f"  {r['scientist']}  --[{r['relation']}]-->  {r['field']}")

## 3. refresh_schema — structured_schema dict

In [ ]:
graph.refresh_schema()
print("node_props keys:", list(graph.structured_schema.get("node_props", {}).keys())[:10])
print("relationships:", graph.structured_schema.get("relationships", [])[:5])

## 4. Idempotency check

`add_graph_documents` uses MERGE internally — adding the same document twice must not
create duplicate edges.

In [ ]:
graph.add_graph_documents([doc])  # second upsert — must not create a duplicate edge
cnt = graph.query(
    "MATCH (a {name: $src})-[r:FOUNDED]->(b {name: $dst}) RETURN count(r) AS cnt",
    params={"src": f"Turing-{tag}", "dst": f"Cryptography-{tag}"},
)
print("FOUNDED edge count after double upsert (expect 1):", cnt[0]["cnt"])

## 5. GraphCypherQAChain — LLM-powered Cypher (optional)

> **This section requires `OPENAI_API_KEY`.** Set it in your environment or via
> `os.environ['OPENAI_API_KEY'] = 'sk-...'` before running.
> The cell is skipped automatically when the key is absent.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print(
        'Skipping: OPENAI_API_KEY is not set. Set it via os.environ["OPENAI_API_KEY"] = "sk-..." and re-run this cell.'
    )
else:
    from langchain.chains import GraphCypherQAChain
    from langchain_openai import ChatOpenAI

    chain = GraphCypherQAChain.from_llm(
        ChatOpenAI(model="gpt-4o-mini", temperature=0),
        graph=graph,
        verbose=True,
        allow_dangerous_requests=True,
    )
    result = chain.invoke(f"Who influenced Shannon-{tag}?")
    print("Answer:", result["result"])

## Cleanup

In [ ]:
# DETACH DELETE atomically removes all edges then the node in one operation.
# Two-step MATCH (n)-[r]-() / DELETE r / DELETE n is avoided because an
# undirected MATCH returns each edge from both endpoints, so the second pass
# fails with "cannot delete node with connected edges".
graph.query("MATCH (n) WHERE n.name ENDS WITH $tag DETACH DELETE n", params={"tag": tag})
print("Cleaned up")
graph.close()
client.close()  # injected client — owned by caller